# Erweiterte Zero-Shot-Klassifikation mit Transformer-Pipelines

In Notebook 2 haben wir die Zero-Shot-Klassifikation genutzt, um Spam zu kennzeichnen. Hier untersuchen wir dieselbe Pipeline umfassender: Wir klassifizieren eigene Sätze mit unseren eigenen Labels, sehen uns an, wie die Methode intern funktioniert, wenden sie auf mehr als 100 Sprachen an und probieren weitere Pipeline-Aufgaben wie die Textgenerierung und die Sentimentanalyse aus.

## Lernziele

Am Ende dieses Notebooks kannst du:

- eine Zero-Shot-Pipeline anwenden, um eigenen Text mit deinen eigenen Kandidatenlabels zu klassifizieren.
- erklären, wie Natural Language Inference die Zero-Shot-Klassifikation möglich macht.
- ein mehrsprachiges Modell nutzen, um Text in mehr als 100 Sprachen zu klassifizieren.
- verschiedene Pipeline-Aufgaben wie die Textgenerierung und die Sentimentanalyse vergleichen.

In [ ]:
from transformers import pipeline

import pandas as pd

In [ ]:
classifier = pipeline(
    "zero-shot-classification", device=0
)  # device=0 nutzt die erste GPU; setze device=-1, um auf der CPU zu rechnen

## Zero-Shot-Klassifikation für eigenen Text

Wir übergeben der Pipeline eine Sequenz und eine Liste von Kandidatenlabels. Standardmäßig geht sie davon aus, dass genau ein Label richtig ist, und gibt für jedes Label einen Score zurück, wobei sich die Scores zu 1 addieren. Die folgenden Beispiele zeigen, wie die Wahl der Labels und sogar die Schreibweise das Ergebnis verändern.

In [ ]:
"""
Wir erstellen eine Funktion, um die Vorhersagen unseres Modells in Tabellenform anzuzeigen
"""


def get_predictions_score(prediction):
    pred_labels = prediction["labels"]
    pred_scores = prediction["scores"]
    seq = [prediction["sequence"]]
    return (
        pd.concat(
            [
                pd.DataFrame(seq),
                pd.DataFrame(pred_labels),
                pd.DataFrame(pred_scores),
            ],
            axis=1,
            ignore_index=True,
        )
        .rename(columns={0: "Sequence", 1: "Labels", 2: "Probability"})
        .set_index(["Sequence"])
    )

In [ ]:
sequence = "Amazon is the longest river in the world"
candidate_labels = ["geography", "delivery"]

pred = classifier(sequence, candidate_labels)
get_predictions_score(pred)

Was passiert, wenn wir die Schreibweise ändern? Hier ändern wir `Amazon` in `amazon`. In diesem Fall macht das kaum einen Unterschied, in anderen Fällen aber schon. Probiere andere Schreibweisen aus, füge Labels hinzu oder entferne welche und beobachte dann, wie sich die Scores verschieben.

In [ ]:
sequence = "amazon is the longest river in the world"
candidate_labels = ["geography", "delivery"]

pred = classifier(sequence, candidate_labels)
get_predictions_score(pred)

Das folgende Beispiel zeigt, wie gut diese Modelle den Kontext nutzen, selbst bei einem kleinen Tippfehler. Probiere andere Schreibweisen aus und beobachte die Ergebnisse.

In [ ]:
sequence = "are we going to Oktoberfest?"
candidate_labels = [
    "food",
    "Munich",
    "bear",
    "wine",
    "pretzel",
    "sausage",
]  ## Was passiert, wenn du bear (Tier) -> beer (Getränk) änderst

pred = classifier(sequence, candidate_labels)
get_predictions_score(pred)

In [ ]:
sequence = "Who are you voting for in 2020?"
candidate_labels = ["food", "public health", "plants", "fruits", "america"]

pred = classifier(sequence, candidate_labels)
get_predictions_score(pred)

> Die Vorhersagen sind hier schlecht, weil die Labels nichts mit der Sequenz zu tun haben. Wir können das verbessern, indem wir Kandidatenlabels angeben, die zum Eingabetext passen.

In [ ]:
## Überlege dir andere Labels, die die Vorhersagen verbessern können
## TIPP: Labels, die mit deinem Text zu tun haben

Für eine Mehrklassenklassifikation übergibst du einfach `multi_class=True`. In diesem Fall sind die Scores unabhängig voneinander, aber jeder Score liegt zwischen 0 und 1.

In [ ]:
sequence = "Who are you voting for in 2020?"
candidate_labels = ["politics", "public health", "economics", "elections"]

pred = classifier(sequence, candidate_labels, multi_label=True)
get_predictions_score(pred)

### Sentimentklassifikation

Hier ist ein Beispiel für eine Sentimentklassifikation: 

In [ ]:
sequence = "I hated this movie. The acting sucked."
candidate_labels = ["positive", "negative"]

pred = classifier(sequence, candidate_labels)
get_predictions_score(pred)

## Wie die Zero-Shot-Klassifikation funktioniert

Das zugrunde liegende Modell ist auf **Natural Language Inference (NLI)** trainiert: Für zwei gegebene Sätze entscheidet es, ob sie einander widersprechen, einander implizieren oder keins von beidem.

Für die Zero-Shot-Klassifikation wird das so angepasst: Der Text, den wir klassifizieren wollen, dient als einer der beiden NLI-Sätze (die Prämisse), und jedes Kandidatenlabel wird zum anderen Satz (der Hypothese). Sagt das Modell vorher, dass die Prämisse die Hypothese impliziert, werten wir das als Vorhersage, dass das Label zutrifft.

Standardmäßig wandelt die Pipeline die Labels mit dem Template `This example is {class_name}.` in Hypothesen um. Das funktioniert in vielen Situationen gut, du kannst es aber anpassen. Fügen wir dem Beispiel zur Sentimentklassifikation eine schwierigere Rezension hinzu:

In [ ]:
sequences = [
    "I hated this movie. The acting sucked.",
    "This movie didn't quite live up to my high expectations, but overall I still really enjoyed it.",
]
candidate_labels = ["positive", "negative"]

classifier(sequences, candidate_labels)

Das zweite Beispiel ist etwas schwieriger. Sehen wir uns an, ob wir die Ergebnisse mit einem Hypothesen-Template verbessern können, das genauer auf die Sentimentanalyse von Rezensionen zugeschnitten ist. Statt des Standard-Templates `This example is {}.` verwenden wir `The sentiment of this review is {}.` (wobei `{}` durch den Namen der Kandidatenklasse ersetzt wird)

In [ ]:
sequences = [
    "I hated this movie. The acting sucked.",
    "This movie didn't quite live up to my high expectations, but overall I still really enjoyed it.",
]
candidate_labels = ["positive", "negative"]
hypothesis_template = "The sentiment of this review is {}."

classifier(sequences, candidate_labels, hypothesis_template=hypothesis_template)

Mit einem präziseren Hypothesen-Template erhalten wir eine genauere Klassifikation der zweiten Rezension.

> Beachte, dass die Sentimentklassifikation hier nur als anschauliches Beispiel dient. Im [Hugging Face Model Hub](https://huggingface.co/models?filter=text-classification) gibt es eine Reihe von Modellen, die speziell für Sentimentaufgaben trainiert wurden und stattdessen verwendet werden können.

### Du bist dran: das Hypothesen-Template optimieren

Die Pipeline wandelt jedes Label mit einem Template in eine Hypothese um. Ein Template, das zur Aufgabe passt, kann die Scores eindeutiger machen. Probiere das an einem Beispiel zur Themenklassifikation aus.

**Schritte**
1. Wähle eine Sequenz (zum Beispiel eine Schlagzeile) und 3 oder 4 Kandidatenlabels.
2. Klassifiziere sie mit dem Standard-Template und dann noch einmal mit einem eigenen `hypothesis_template` wie `"This news article is about {}."`.
3. Vergleiche die Scores aus beiden Aufrufen.

**Tipp:** Übergib `hypothesis_template=` als Keyword-Argument an `classifier(...)` und verwende `get_predictions_score(...)` erneut, um jedes Ergebnis anzuzeigen.

**Erwartetes Ergebnis:** Das eigene, aufgabenspezifische Template gibt dem richtigen Label meist einen höheren Score.

In [ ]:
sequence = "The central bank raised interest rates again this quarter."
candidate_labels = ["economics", "sports", "politics", "weather"]

# TODO 1: mit dem Standard-Template klassifizieren
default_pred = ...

# TODO 2: noch einmal mit einem eigenen Hypothesen-Template klassifizieren
custom_pred = classifier(sequence, candidate_labels, hypothesis_template=...)

# TODO 3: beide Ergebnisse anzeigen und die Scores vergleichen
display(get_predictions_score(default_pred))
display(get_predictions_score(custom_pred))

## Zero-Shot-Klassifikation in mehr als 100 Sprachen


Möchtest du die Pipeline für andere Sprachen als Englisch nutzen? Es gibt ein sprachübergreifendes Modell auf Basis von XLM RoBERTa, das du verwenden kannst, indem du beim Erstellen der Pipeline `model='joeddav/xlm-roberta-large-xnli'` übergibst: 

In [ ]:
classifier = pipeline(
    "zero-shot-classification", model="joeddav/xlm-roberta-large-xnli", device=0
)  # device=0 nutzt die erste GPU; setze device=-1, um auf der CPU zu rechnen

Du kannst es mit jeder beliebigen Kombination von Sprachen verwenden. Klassifizieren wir zum Beispiel einen russischen Satz mit englischen Kandidatenlabels:

In [ ]:
sequence = "За кого вы голосуете в 2020 году?"  # Übersetzung: „Wen wählst du 2020?“
candidate_labels = ["Europe", "public health", "politics"]

classifier(sequence, candidate_labels)

Machen wir jetzt dasselbe, aber mit den Labels auf Französisch:



In [ ]:
sequence = "За кого вы голосуете в 2020 году?"  # Übersetzung: „Wen wählst du 2020?“
candidate_labels = ["Europe", "santé publique", "politique"]

classifier(sequence, candidate_labels)

Wie im letzten Abschnitt besprochen, ist das Standard-Hypothesen-Template das englische `This text is {}.`. Wenn du nur in einer einzigen Sprache arbeitest, kann es sich lohnen, es in diese Sprache zu übersetzen:

In [ ]:
sequence = "¿A quién vas a votar en 2020?"
candidate_labels = ["Europa", "salud pública", "política"]
hypothesis_template = "Este ejemplo es {}."

classifier(sequence, candidate_labels, hypothesis_template=hypothesis_template)

Das Modell ist auf XNLI feinabgestimmt, das 15 Sprachen umfasst: Arabisch, Bulgarisch, Chinesisch, Englisch, Französisch, Deutsch, Griechisch, Hindi, Russisch, Spanisch, Swahili, Thai, Türkisch, Urdu und Vietnamesisch. Das Basismodell wurde auf 85 weiteren Sprachen trainiert, daher funktioniert das Modell bis zu einem gewissen Grad für jede Sprache im Trainingskorpus von XLM RoBERTa (die vollständige Liste findest du in Anhang A des [Papers zu XLM RoBERTa](https://arxiv.org/abs/1911.02116)).

Mehr dazu findest du auf der [Modellseite](https://huggingface.co/joeddav/xlm-roberta-large-xnli).

## Weitere Pipeline-Aufgaben

Die Zero-Shot-Klassifikation ist nur eine von vielen Aufgaben, die die Funktion `pipeline` unterstützt. Über dieselbe Schnittstelle bekommst du Textgenerierung, Sentimentanalyse, Übersetzung, Textzusammenfassung und mehr, jeweils auf Basis eines passenden vortrainierten Modells.

[Hier kannst du nachlesen](https://huggingface.co/docs/transformers/main_classes/pipelines), welche verschiedenen Modelle über die Pipeline von Hugging Face verfügbar sind.

### Textgenerierung

In [ ]:
text_gen = pipeline("text-generation", model="gpt2")  # um die GPU zu nutzen

In [ ]:
prompt = "Data Science is"
text_gen(prompt, max_length=30, num_return_sequences=3)

Du kannst verschiedene Anfangssätze ausprobieren. Wenn du kürzere oder längere Sätze möchtest, kannst du das Argument `max_length` ändern.

### Sentimentanalyse

Das Beispiel zur Sentimentanalyse vom Anfang des Notebooks lässt sich auch mit einer Pipeline für die Sentimentanalyse umsetzen

### Du bist dran: eine spezialisierte Pipeline für die Sentimentanalyse

Zero-Shot ist flexibel, aber ein Modell, das für die Aufgabe trainiert wurde, ist einfacher und oft besser. Erstelle eine `sentiment-analysis`-Pipeline und bewerte damit ein paar Sätze.

**Schritte**
1. Erstelle `pipeline("sentiment-analysis")` (keine Kandidatenlabels nötig, das Modell ist bereits für Sentiment trainiert).
2. Führe sie für zwei Sätze aus, einen eindeutig positiven und einen eindeutig negativen.
3. Sieh dir das Label und den Score an, die für jeden Satz zurückgegeben werden.

**Tipp:** Die Sentiment-Pipeline gibt eine Liste von Dictionaries wie `[{'label': 'POSITIVE', 'score': 0.99}]` zurück. Du übergibst keine `candidate_labels`.

**Erwartetes Ergebnis:** Der positive Satz wird mit hoher Konfidenz als POSITIVE eingestuft, der negative als NEGATIVE.

In [ ]:
# TODO 1: eine Pipeline für die Sentimentanalyse erstellen
sentiment = ...

reviews = [
    "I absolutely loved this film, it was fantastic.",
    "A complete waste of time, terrible from start to finish.",
]
# TODO 2: die Pipeline für die Rezensionen ausführen, dann die Ergebnisse ausgeben
results = ...
print(results)

## Optional: in den Hugging Face Hub hochladen

Du kannst ein Konto bei Hugging Face und ein Zugriffstoken erstellen, wenn du Inhalte in einem Repository erstellen oder dorthin hochladen möchtest (zum Beispiel, wenn du ein Modell trainierst oder eine Model Card bearbeitest).

- Erstelle ein Konto unter https://huggingface.co/
- Gehe nach der Anmeldung zu **Settings -> Access Tokens**
- Erstelle ein neues Token und gib ihm Schreibrechte
- Führe diese Befehle aus:
    - `brew install huggingface-cli`
    - `huggingface-cli login` und füge das Zugriffstoken von Hugging Face ein
    - Füge kein Zugriffstoken für GitHub hinzu, falls danach gefragt wird

Quelle: https://huggingface.co/docs/hub/security-tokens

## Zusammenfassung

In diesem Notebook hast du:

- eine Zero-Shot-Pipeline auf eigene Sätze mit deinen eigenen Labels angewendet.
- erklärt, wie Natural Language Inference die Labels in Hypothesen umwandelt, um die Zero-Shot-Klassifikation zu ermöglichen.
- ein mehrsprachiges Modell genutzt, um Text in vielen Sprachen zu klassifizieren.
- weitere Pipeline-Aufgaben verglichen, darunter die Textgenerierung und die Sentimentanalyse.

## Quellen & weiterführende Literatur

- [**Hugging Face: Pipelines**](https://huggingface.co/docs/transformers/main_classes/pipelines): Referenz für alle Pipeline-Aufgaben, die in diesem Notebook verwendet werden.
- [**Hugging Face: Zero-Shot Classification**](https://huggingface.co/tasks/zero-shot-classification): Hintergrund zur Aufgabe und zu den Modellen, die sie unterstützen.